# 1D Recursive Polynomial KAN Approximation

An exploratory prototype for fitting a noisy piecewise target function with a learnable recursive polynomial basis. This notebook is not the Stieltjes-Wigert implementation; it is retained as an earlier experiment.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# =========================
# 1. Target function
# =========================

def target_function(x_np):
    """
    x_np: numpy array, shape (N,)
    returns y_np: same shape
    """
    y = np.zeros_like(x_np)
    mask1 = x_np < 0.5
    y[mask1] = np.sin(20 * np.pi * x_np[mask1]) + x_np[mask1] ** 2
    mask2 = (0.5 <= x_np) & (x_np < 1.5)
    y[mask2] = 0.5 * x_np[mask2] * np.exp(-x_np[mask2]) + np.abs(np.sin(5 * np.pi * x_np[mask2]))
    mask3 = x_np >= 1.5
    y[mask3] = np.log(x_np[mask3] - 1) / np.log(2) - np.cos(2 * np.pi * x_np[mask3])

    noise = np.random.normal(0, 0.2, y.shape)
    y += noise
    return y

# =========================
# 2. Recursive polynomial basis
# =========================

class RecursivePolyBasis(nn.Module):
    """
    1D basis:
        R_0(x) = 0
        R_1(x) = 1
        R_{n+1}(x) = (a x^2 + b x + c) R_n(x) + (d x + e) R_{n-1}(x)
    y = sum_{n=0}^K w_n R_n(x)
    """
    def __init__(self, K, param_bound=3.0, init_like_cheb=True):
        super().__init__()
        self.K = K
        self.param_bound = param_bound

        self.a_raw = nn.Parameter(torch.zeros(1))
        self.b_raw = nn.Parameter(torch.zeros(1))
        self.c_raw = nn.Parameter(torch.zeros(1))
        self.d_raw = nn.Parameter(torch.zeros(1))
        self.e_raw = nn.Parameter(torch.zeros(1))

        self.w = nn.Parameter(torch.zeros(K + 1))

        if init_like_cheb:
            with torch.no_grad():
                self.a_raw.fill_(0.0)
                self.b_raw.fill_(0.5)
                self.c_raw.fill_(0.0)
                self.d_raw.fill_(0.0)
                self.e_raw.fill_(-0.5)
                self.w.normal_(mean=0.0, std=0.01)

    def _squash_params(self):
        a = self.param_bound * torch.tanh(self.a_raw)
        b = self.param_bound * torch.tanh(self.b_raw)
        c = self.param_bound * torch.tanh(self.c_raw)
        d = self.param_bound * torch.tanh(self.d_raw)
        e = self.param_bound * torch.tanh(self.e_raw)
        return a, b, c, d, e

    def forward(self, x):
        """
        x: (batch,1) or (batch,)
        """
        if x.dim() > 1:
            x = x.squeeze(-1)

        # The target domain is approximately [0, 2]; clip for stability.
        x = x.clamp(0.0, 2.0)

        a, b, c, d, e = self._squash_params()

        R0 = torch.zeros_like(x)   # R_0
        R1 = torch.ones_like(x)    # R_1
        Rs = [R0, R1]

        for n in range(1, self.K):
            coef1 = a * x**2 + b * x + c
            coef2 = d * x + e
            R_next = coef1 * Rs[-1] + coef2 * Rs[-2]
            Rs.append(R_next)

        R_stack = torch.stack(Rs, dim=-1)  # (batch, K+1)
        y = (R_stack * self.w).sum(dim=-1)
        return y.unsqueeze(-1)  # (batch,1)

# =========================
# 3. Simple 1D model
# =========================

class PolyKAN1D(nn.Module):
    def __init__(self, K):
        super().__init__()
        self.lin = nn.Linear(1, 1)
        self.basis = RecursivePolyBasis(K=K, init_like_cheb=True)

    def forward(self, x):
        h = self.lin(x)           # (batch,1)
        h = h.squeeze(-1)         # (batch,)
        y = self.basis(h)         # (batch,1)
        return y

# =========================
# 4. Train on the piecewise noisy function
# =========================

def train_on_piecewise():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # data on [0, 2]
    n_samples = 5000
    x_np = np.random.uniform(0.0, 2.0, size=(n_samples,))
    y_np = target_function(x_np)

    x = torch.from_numpy(x_np).float().unsqueeze(-1).to(device)
    y = torch.from_numpy(y_np).float().unsqueeze(-1).to(device)

    # hyperparams
    K = 10           # basis degree (0..K)
    n_epochs = 3000
    batch_size = 256
    lr = 1e-3

    model = PolyKAN1D(K=K).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    loss_fn = nn.MSELoss()

    for epoch in range(1, n_epochs + 1):
        perm = torch.randperm(n_samples, device=device)
        x = x[perm]
        y = y[perm]

        for i in range(0, n_samples, batch_size):
            xb = x[i:i+batch_size]
            yb = y[i:i+batch_size]

            pred = model(xb)
            loss = loss_fn(pred, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        scheduler.step()

        if epoch % 300 == 0:
            print(f"Epoch {epoch}/{n_epochs}, loss = {loss.item():.5f}")

    # evaluation on a grid
    x_test_np = np.linspace(0.0, 2.0, 800)
    y_test_np = target_function(x_test_np)

    x_test = torch.from_numpy(x_test_np).float().unsqueeze(-1).to(device)
    with torch.no_grad():
        y_pred = model(x_test).cpu().numpy().squeeze()

    # plot
    plt.figure(figsize=(7,4))
    plt.scatter(x_np, y_np, s=5, alpha=0.3, label='noisy samples')
    plt.plot(x_test_np, y_test_np, label='true (noisy) function', color='gray', alpha=0.7)
    plt.plot(x_test_np, y_pred, label='RecursivePoly-KAN approx', color='red')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    train_on_piecewise()
